# Silver Lake Explorer (EDA Ready)

Notebook para explorar datos de `datalake_silver`, consolidarlos y exportar un CSV limpio para EDA antes de construir la capa Gold.

In [4]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display

def find_project_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for candidate in [start, *start.parents]:
        if (candidate / 'README.md').exists() and (candidate / 'datalake_silver').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root()
SILVER_DIR = PROJECT_ROOT / 'datalake_silver'
OUTPUT_DIR = PROJECT_ROOT / 'notebooks' / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SILVER_EDA_CSV = OUTPUT_DIR / 'silver_eda.csv'

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'SILVER_DIR = {SILVER_DIR}')
print(f'SILVER_EDA_CSV = {SILVER_EDA_CSV}')

PROJECT_ROOT = /home/naciscric/Documents/university/2026-1/Data-Analysis-Programming
SILVER_DIR = /home/naciscric/Documents/university/2026-1/Data-Analysis-Programming/datalake_silver
SILVER_EDA_CSV = /home/naciscric/Documents/university/2026-1/Data-Analysis-Programming/notebooks/output/silver_eda.csv


In [5]:
silver_files = sorted(SILVER_DIR.glob('*.parquet'))
print(f'Parquet files found: {len(silver_files)}')
display(pd.DataFrame({'silver_parquet': [f.name for f in silver_files]}))

Parquet files found: 9


,silver_parquet
0,noticias_processed_20260411_034949.parquet
1,noticias_processed_20260524_202336.parquet
2,noticias_processed_20260524_202358.parquet
3,noticias_processed_20260524_202512.parquet
4,tweets_processed_20260411_034949.parquet
5,tweets_processed_20260524_202334.parquet
6,tweets_processed_20260524_202358.parquet
7,tweets_processed_20260524_202510.parquet
8,tweets_processed_20260524_202514.parquet


In [6]:
def infer_silver_source(path: Path) -> str:
    name = path.name.lower()
    if 'tweets' in name:
        return 'tweets'
    if 'noticias' in name or 'news' in name:
        return 'news'
    return 'unknown'

frames = []
for file_path in silver_files:
    df = pd.read_parquet(file_path)
    df['source_file'] = file_path.name
    df['source_group'] = infer_silver_source(file_path)
    frames.append(df)

if frames:
    silver_raw = pd.concat(frames, ignore_index=True)
else:
    silver_raw = pd.DataFrame()

print(f'silver_raw shape: {silver_raw.shape}')
silver_raw.head(5)

silver_raw shape: (33, 11)


,_id,date,news,createdAt,updatedAt,__v,source_file,source_group,tweets,has_next_page,next_cursor
0,69d947bb4dd967352a8ee984,1.775847e+12,"[{'_id': '69d947bb4dd967352a8ee985', 'comments...",2026-04-10 18:55:55.421,2026-04-10 18:55:55.421,0,noticias_processed_20260411_034949.parquet,news,NaN,NaN,NaN
1,6a134f02295795246a978b1a,1.779650e+12,"[{'_id': '6a134f02295795246a978b1b', 'comments...",2026-05-24 19:18:26.734,2026-05-24 19:18:26.734,0,noticias_processed_20260524_202336.parquet,news,NaN,NaN,NaN
2,6a134f02295795246a978b1a,1.779650e+12,"[{'_id': '6a134f02295795246a978b1b', 'comments...",2026-05-24 19:18:26.734,2026-05-24 19:18:26.734,0,noticias_processed_20260524_202358.parquet,news,NaN,NaN,NaN
3,6a134f02295795246a978b1a,1.779650e+12,"[{'_id': '6a134f02295795246a978b1b', 'comments...",2026-05-24 19:18:26.734,2026-05-24 19:18:26.734,0,noticias_processed_20260524_202512.parquet,news,NaN,NaN,NaN
4,69d86f08aae727ea3c009a31,NaN,NaN,2026-04-10 03:31:20.837,2026-04-10 03:31:20.837,0,tweets_processed_20260411_034949.parquet,tweets,"[{'article': None, 'author': {'automatedBy': N...",True,DAADDAABCgABHFg4fD4bQbEKAAIcSlokVJpQ7AAIAAIAAA...


## Auditoria de columnas

Revisamos tipos, nulos y cardinalidad para decidir columnas utiles para EDA/Gold.

In [7]:
def is_missing_value(value) -> bool:
    try:
        return bool(pd.isna(value))
    except Exception:
        return False


def safe_nunique(series: pd.Series) -> int:
    normalized = series.map(lambda value: None if is_missing_value(value) else repr(value))
    return int(normalized.dropna().nunique(dropna=True))


if silver_raw.empty:
    audit = pd.DataFrame(columns=['column', 'dtype', 'missing_ratio', 'non_null_count', 'nunique'])
else:
    audit = pd.DataFrame({
        'column': silver_raw.columns,
        'dtype': silver_raw.dtypes.astype(str).values,
        'missing_ratio': silver_raw.isna().mean().values,
        'non_null_count': silver_raw.notna().sum().values,
        'nunique': [safe_nunique(silver_raw[c]) for c in silver_raw.columns],
    }).sort_values(['missing_ratio', 'column'], ascending=[False, True])

display(audit.head(80))

,column,dtype,missing_ratio,non_null_count,nunique
1,date,float64,0.878788,4,2
2,news,object,0.878788,4,2
10,next_cursor,str,0.484848,17,4
9,has_next_page,object,0.121212,29,2
8,tweets,object,0.121212,29,5
5,__v,int64,0.000000,33,1
0,_id,str,0.000000,33,9
3,createdAt,datetime64[ns],0.000000,33,9
6,source_file,str,0.000000,33,9
7,source_group,str,0.000000,33,2


In [13]:
try:
    from langdetect import DetectorFactory, LangDetectException, detect
    DetectorFactory.seed = 42
    LANGDETECT_AVAILABLE = True
except Exception:
    LANGDETECT_AVAILABLE = False


def first_existing(columns: list[str], candidates: list[str]) -> list[str]:
    available = set(columns)
    return [c for c in candidates if c in available]


def clean_text(text: str | None) -> str:
    if text is None:
        return ''
    cleaned = str(text)
    cleaned = re.sub(r'https?://\S+|www\.\S+', ' ', cleaned)
    cleaned = re.sub(r'@\w+', ' ', cleaned)
    cleaned = re.sub(r'#', ' ', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned)
    return cleaned.strip()


def detect_language_safe(text: str) -> str:
    if not text:
        return 'unknown'
    if not LANGDETECT_AVAILABLE:
        return 'unknown'
    try:
        return detect(text)
    except LangDetectException:
        return 'unknown'


def to_text(value) -> str:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ''
    if isinstance(value, str):
        return value
    if isinstance(value, (list, tuple, set)):
        return ' '.join(str(v) for v in value if v is not None)
    if hasattr(value, 'tolist') and not isinstance(value, (str, bytes)):
        as_list = value.tolist()
        if isinstance(as_list, list):
            return ' '.join(str(v) for v in as_list if v is not None)
        return str(as_list)
    return str(value)


def parse_mixed_datetime(series: pd.Series) -> pd.Series:
    as_text = series.map(to_text)
    numeric = pd.to_numeric(as_text, errors='coerce')
    numeric_ratio = float(numeric.notna().mean()) if len(numeric) else 0.0
    if numeric_ratio > 0.8:
        return pd.to_datetime(numeric, errors='coerce', unit='ms')
    return pd.to_datetime(as_text, errors='coerce', format='mixed')


def flatten_silver(df: pd.DataFrame) -> pd.DataFrame:
    parts = []

    if 'tweets' in df.columns:
        tweet_base = df[df['tweets'].notna()].copy()
        if not tweet_base.empty:
            tweet_base = tweet_base.explode('tweets', ignore_index=True)
            tweet_info = pd.json_normalize(tweet_base['tweets'], sep='_').add_prefix('tweet_')
            tweet_flat = pd.concat([tweet_base.drop(columns=['tweets']), tweet_info], axis=1)
            tweet_flat['record_type'] = 'tweet'
            parts.append(tweet_flat)

    if 'news' in df.columns:
        news_base = df[df['news'].notna()].copy()
        if not news_base.empty:
            news_base = news_base.explode('news', ignore_index=True)
            news_info = pd.json_normalize(news_base['news'], sep='_').add_prefix('news_')
            news_flat = pd.concat([news_base.drop(columns=['news']), news_info], axis=1)
            news_flat['record_type'] = 'news'
            parts.append(news_flat)

            if 'news_comments' in news_flat.columns:
                comments_flat = news_flat.explode('news_comments', ignore_index=True).copy()
                comments_flat['comment_text'] = comments_flat['news_comments'].map(to_text)
                comments_flat['record_type'] = 'news_comment'
                parts.append(comments_flat)

    if not parts:
        fallback = df.copy()
        fallback['record_type'] = 'snapshot'
        return fallback

    return pd.concat(parts, ignore_index=True)


silver_expanded = flatten_silver(silver_raw)
print(f'silver_expanded shape: {silver_expanded.shape}')

text_candidates = [
    'tweet_text', 'comment_text', 'news_comments', 'news_newsLink', 'text',
    'clean_text', 'comments', 'title', 'headline', 'content', 'news_title', 'news_text'
]
date_candidates = [
    'createdAt', 'updatedAt', 'date', 'date_parsed', 'tweet_createdAt',
    'tweet_created_at', 'published_at', 'timestamp'
]
id_candidates = [
    'id', '_id', 'tweet_id', 'news_id', 'news__id', 'source_file', 'record_type'
]

base_cols = first_existing(list(silver_expanded.columns), id_candidates)
base_cols += first_existing(list(silver_expanded.columns), date_candidates)
base_cols += first_existing(list(silver_expanded.columns), text_candidates)

metric_cols = [
    c for c in silver_expanded.columns
    if any(k in c.lower() for k in ['like', 'reply', 'retweet', 'quote', 'view', 'score', 'sentiment'])
]
meta_cols = [
    c for c in silver_expanded.columns
    if any(k in c.lower() for k in ['author', 'user', 'brand', 'source', 'lang'])
]

keep_cols = list(dict.fromkeys(base_cols + metric_cols + meta_cols))
silver_eda = silver_expanded[keep_cols].copy() if keep_cols else silver_expanded.copy()

text_cols_existing = first_existing(list(silver_eda.columns), text_candidates)
if text_cols_existing:
    text_frame = silver_eda[text_cols_existing].copy().map(to_text)
    silver_eda['raw_text'] = text_frame.replace('', pd.NA).bfill(axis=1).iloc[:, 0].fillna('').astype('string')
else:
    object_like_cols = [c for c in silver_eda.columns if silver_eda[c].dtype == 'object']
    if object_like_cols:
        fallback = silver_eda[object_like_cols[0]].map(to_text)
        silver_eda['raw_text'] = fallback.astype('string')
    else:
        silver_eda['raw_text'] = ''

silver_eda['clean_text'] = silver_eda['raw_text'].map(clean_text)
silver_eda['text_length'] = silver_eda['clean_text'].str.len()
silver_eda['detected_lang'] = silver_eda['clean_text'].fillna('').map(detect_language_safe)
silver_eda['has_text'] = silver_eda['clean_text'].str.len().fillna(0).gt(0)

date_cols_existing = first_existing(list(silver_eda.columns), date_candidates)
if date_cols_existing:
    parsed_series = None
    for col in date_cols_existing:
        candidate = parse_mixed_datetime(silver_eda[col])
        parsed_series = candidate if parsed_series is None else parsed_series.fillna(candidate)
    silver_eda['parsed_datetime'] = parsed_series
    silver_eda['event_date'] = silver_eda['parsed_datetime'].dt.date.astype('string')
else:
    silver_eda['parsed_datetime'] = pd.NaT
    silver_eda['event_date'] = pd.Series([''] * len(silver_eda), dtype='string')

dedupe_keys = [c for c in ['source_file', 'record_type', 'tweet_id', 'news_id', 'news__id', '_id', 'id', 'clean_text', 'event_date'] if c in silver_eda.columns]
before_dedup = len(silver_eda)
if dedupe_keys:
    silver_eda = silver_eda.drop_duplicates(subset=dedupe_keys).reset_index(drop=True)
after_dedup = len(silver_eda)

print(f'Rows before dedup: {before_dedup}')
print(f'Rows after dedup: {after_dedup}')
print(f'Columns in silver_eda: {len(silver_eda.columns)}')
print(f'Rows with text: {int(silver_eda["has_text"].sum())}')
silver_eda.head(5)

silver_expanded shape: (1156, 192)
Rows before dedup: 1156
Rows after dedup: 1156
Columns in silver_eda: 173
Rows with text: 1144


,_id,tweet_id,news__id,source_file,record_type,createdAt,updatedAt,date,tweet_createdAt,tweet_text,...,tweet_card_user_refs_results,tweet_author_profile_bio_entities_description,tweet_author_profile_bio_entities,raw_text,clean_text,text_length,detected_lang,has_text,parsed_datetime,event_date
0,69d86f08aae727ea3c009a31,2042444537281593777,NaN,tweets_processed_20260411_034949.parquet,tweet,2026-04-10 03:31:20.837,2026-04-10 03:31:20.837,NaN,Fri Apr 10 03:28:19 +0000 2026,Hey iPhone battery replacers! \n\niPhone 13 - ...,...,NaN,NaN,NaN,Hey iPhone battery replacers! \n\niPhone 13 - ...,Hey iPhone battery replacers! iPhone 13 - 4 ye...,144,en,True,2026-04-10 03:31:20.837,2026-04-10
1,69d86f08aae727ea3c009a31,2042356352027410907,NaN,tweets_processed_20260411_034949.parquet,tweet,2026-04-10 03:31:20.837,2026-04-10 03:31:20.837,NaN,Thu Apr 09 21:37:54 +0000 2026,@SamsungIndia @SamsungSupport\nVery disappoint...,...,NaN,NaN,NaN,@SamsungIndia @SamsungSupport\nVery disappoint...,Very disappointed. I bought a Samsung Galaxy S...,238,en,True,2026-04-10 03:31:20.837,2026-04-10
2,69d86f08aae727ea3c009a31,2041877436258726386,NaN,tweets_processed_20260411_034949.parquet,tweet,2026-04-10 03:31:20.837,2026-04-10 03:31:20.837,NaN,Wed Apr 08 13:54:51 +0000 2026,After using the Samsung Galaxy S25 FE for over...,...,NaN,NaN,NaN,After using the Samsung Galaxy S25 FE for over...,After using the Samsung Galaxy S25 FE for over...,519,en,True,2026-04-10 03:31:20.837,2026-04-10
3,69d86f08aae727ea3c009a31,2041654754456301988,NaN,tweets_processed_20260411_034949.parquet,tweet,2026-04-10 03:31:20.837,2026-04-10 03:31:20.837,NaN,Tue Apr 07 23:10:00 +0000 2026,The PELADN WO4 AMD Ryzen 5 Mini PC is an amaz...,...,NaN,NaN,NaN,The PELADN WO4 AMD Ryzen 5 Mini PC is an amaz...,The PELADN WO4 AMD Ryzen 5 Mini PC is an amazi...,232,en,True,2026-04-10 03:31:20.837,2026-04-10
4,69d86f08aae727ea3c009a31,2041072528534495498,NaN,tweets_processed_20260411_034949.parquet,tweet,2026-04-10 03:31:20.837,2026-04-10 03:31:20.837,NaN,Mon Apr 06 08:36:26 +0000 2026,- Nokia 7210 Supernova (loved the speaker and ...,...,NaN,NaN,NaN,- Nokia 7210 Supernova (loved the speaker and ...,- Nokia 7210 Supernova (loved the speaker and ...,279,en,True,2026-04-10 03:31:20.837,2026-04-10


In [14]:
silver_eda.to_csv(SILVER_EDA_CSV, index=False, encoding='utf-8')
print(f'Saved CSV: {SILVER_EDA_CSV}')
print(f'Rows exported: {len(silver_eda)}')
print(f'Columns exported: {len(silver_eda.columns)}')

Saved CSV: /home/naciscric/Documents/university/2026-1/Data-Analysis-Programming/notebooks/output/silver_eda.csv
Rows exported: 1156
Columns exported: 173


## Resumen rapido para EDA y proximo paso Gold

Con este CSV puedes hacer EDA (nulos, distribuciones, outliers, segmentos por fuente/fecha) y luego definir reglas para construir la capa Gold con features estables.

In [10]:
summary = {
    'rows': len(silver_eda),
    'columns': len(silver_eda.columns),
    'non_empty_text_ratio': float(silver_eda['has_text'].mean()) if 'has_text' in silver_eda.columns and len(silver_eda) else 0.0,
}
print(summary)

if 'detected_lang' in silver_eda.columns:
    display(silver_eda['detected_lang'].value_counts().head(10).to_frame('rows'))

if 'source_file' in silver_eda.columns:
    display(silver_eda['source_file'].value_counts().head(10).to_frame('rows'))

missing_top = silver_eda.isna().mean().sort_values(ascending=False).head(20).to_frame('missing_ratio')
display(missing_top)

{'rows': 1156, 'columns': 173, 'non_empty_text_ratio': 0.9896193771626297}


,rows
detected_lang,
en,1123
unknown,12
tl,4
de,3
so,3
pt,3
et,3
sl,1
fr,1


,rows
source_file,
noticias_processed_20260411_034949.parquet,249
noticias_processed_20260524_202336.parquet,185
noticias_processed_20260524_202358.parquet,185
noticias_processed_20260524_202512.parquet,185
tweets_processed_20260524_202334.parquet,83
tweets_processed_20260524_202358.parquet,83
tweets_processed_20260524_202510.parquet,83
tweets_processed_20260524_202514.parquet,83
tweets_processed_20260411_034949.parquet,20


,missing_ratio
tweet_quoted_tweet_author_automatedBy,1.0
tweet_retweeted_tweet,1.0
tweet_quoted_tweet_article,1.0
tweet_quoted_tweet,1.0
tweet_inReplyToId,1.0
tweet_quoted_tweet_quoted_tweet_retweeted_tweet,1.0
tweet_quoted_tweet_quoted_tweet_article,1.0
tweet_quoted_tweet_author_verifiedType,1.0
tweet_quoted_tweet_inReplyToId,1.0
tweet_quoted_tweet_inReplyToUserId,1.0


## Validacion de nulos: estructurales vs reales

Estas celdas verifican si los nulos vienen por columnas no aplicables segun el tipo de registro (`tweet`, `news`, `news_comment`) o si son faltantes reales en campos que deberian existir.

In [11]:
source_level_quality = (
    silver_raw.groupby('source_file')
    .agg(
        rows=('source_file', 'size'),
        non_null_tweets=('tweets', lambda s: int(s.notna().sum()) if 'tweets' in silver_raw.columns else 0),
        non_null_news=('news', lambda s: int(s.notna().sum()) if 'news' in silver_raw.columns else 0),
    )
    .reset_index()
)

display(source_level_quality)

record_type_quality = (
    silver_eda.groupby('record_type')
    .agg(
        rows=('record_type', 'size'),
        has_text_ratio=('has_text', 'mean'),
        null_text_ratio=('raw_text', lambda s: float((s.fillna('') == '').mean())),
        null_datetime_ratio=('parsed_datetime', lambda s: float(s.isna().mean())),
    )
    .reset_index()
)

display(record_type_quality)

core_fields = [c for c in ['tweet_id', 'news__id', 'comment_text', 'tweet_text', 'news_newsLink'] if c in silver_eda.columns]
if core_fields:
    core_nulls = (
        silver_eda.groupby('record_type')[core_fields]
        .apply(lambda df: df.isna().mean())
        .reset_index()
    )
    display(core_nulls)

,source_file,rows,non_null_tweets,non_null_news
0,noticias_processed_20260411_034949.parquet,1,0,1
1,noticias_processed_20260524_202336.parquet,1,0,1
2,noticias_processed_20260524_202358.parquet,1,0,1
3,noticias_processed_20260524_202512.parquet,1,0,1
4,tweets_processed_20260411_034949.parquet,1,1,0
5,tweets_processed_20260524_202334.parquet,7,7,0
6,tweets_processed_20260524_202358.parquet,7,7,0
7,tweets_processed_20260524_202510.parquet,7,7,0
8,tweets_processed_20260524_202514.parquet,7,7,0


,record_type,rows,has_text_ratio,null_text_ratio,null_datetime_ratio
0,news,39,1.000000,0.0,0.0
1,news_comment,765,1.000000,0.0,0.0
2,tweet,352,0.965909,0.034091,0.0


,record_type,tweet_id,news__id,comment_text,tweet_text,news_newsLink
0,news,1.000000,0.0,1.0,1.000000,0.0
1,news_comment,1.000000,0.0,0.0,1.000000,0.0
2,tweet,0.034091,1.0,1.0,0.034091,1.0


In [12]:
files_reloaded = []
for file_path in silver_files:
    test_df = pd.read_parquet(file_path)
    files_reloaded.append({
        'source_file': file_path.name,
        'rows_read': len(test_df),
        'columns_read': len(test_df.columns),
        'read_ok': True,
    })

read_validation = pd.DataFrame(files_reloaded)

total_rows_raw = int(silver_raw.groupby('source_file').size().sum())
total_rows_reloaded = int(read_validation['rows_read'].sum())

print(f'Total rows from concatenated raw: {total_rows_raw}')
print(f'Total rows from direct reload: {total_rows_reloaded}')
print(f'Row count match: {total_rows_raw == total_rows_reloaded}')

display(read_validation.sort_values('source_file'))

Total rows from concatenated raw: 33
Total rows from direct reload: 33
Row count match: True


,source_file,rows_read,columns_read,read_ok
0,noticias_processed_20260411_034949.parquet,1,6,True
1,noticias_processed_20260524_202336.parquet,1,6,True
2,noticias_processed_20260524_202358.parquet,1,6,True
3,noticias_processed_20260524_202512.parquet,1,6,True
4,tweets_processed_20260411_034949.parquet,1,7,True
5,tweets_processed_20260524_202334.parquet,7,7,True
6,tweets_processed_20260524_202358.parquet,7,7,True
7,tweets_processed_20260524_202510.parquet,7,7,True
8,tweets_processed_20260524_202514.parquet,7,7,True
